# 08 - Explainable AI with SHAP

Understand what drives stock forecasts.


## Interpreting Forecast Drivers

**Definition**  
Explainability quantifies feature contribution to predictions.

**Theory**  
SHAP uses cooperative game theory to attribute each feature's marginal contribution.

**Mathematical Intuition**  
Prediction is decomposed as `f(x)=ϕ0 + Σϕi`.

**Financial Intuition**  
Feature attribution helps validate whether model logic aligns with market intuition.

**Business Impact**  
Supports model risk governance, debugging, and stakeholder trust.

**Real-World Example**  
If volume shock dominates predictions during stress periods, model may be liquidity-sensitive.

**Visual Explanation**  
Code cells below generate real plots from Apple market data.

**Code Explanation**  
Each code block is annotated inline and uses shared production modules from `src/`.

**Interpretation of Results**  
After each output, interpret what signal quality, risk, and forecasting reliability imply.


In [ ]:

import sys
import os
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = (PROJECT_ROOT / '..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.forecast_pipeline import ForecastingFramework
from src.data_loader import load_stock_data, split_data
from src.features import create_features
from src.evaluation import regression_metrics
from src.visualization import *

OUT = PROJECT_ROOT / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = PROJECT_ROOT / 'config' / 'config.yaml'
FAST_NOTEBOOK_MODE = os.getenv('NOTEBOOK_FULL_RUN', '0') != '1'

def make_framework():
    framework = ForecastingFramework(str(CONFIG_PATH))
    if FAST_NOTEBOOK_MODE:
        framework.config['models']['automl']['lazypredict'] = False
        framework.config['models']['automl']['pycaret'] = False
        framework.config['models']['automl']['flaml'] = False
        framework.config['models']['deep_learning']['epochs'] = 8
        framework.config['models']['deep_learning']['early_stopping_patience'] = 3
        framework.config['weight_optimization']['methods'] = ['grid']
    return framework

framework = make_framework()


In [ ]:

from src.feature_importance import save_shap_summary_plot, save_shap_dependence_plot

framework = make_framework()
framework.load_data()
base = framework.train_baselines(horizon=1)

best_model_name = base['leaderboard'].iloc[0]['model']
best_model = base['results'][best_model_name].model

explain_out = framework.explain(horizon=1, model=best_model)
print('Best model:', best_model_name)
print(explain_out['summary'])

if explain_out['shap_available']:
    ds = framework.prepare_horizon_dataset(1)
    shap_path = OUT / 'plots/08_shap_summary.png'
    dep_path = OUT / 'plots/08_shap_dependence_close_lag_1.png'
    save_shap_summary_plot(explain_out['shap_values'], ds.X_test[:200], shap_path)
    save_shap_dependence_plot(explain_out['shap_values'], ds.X_test[:200], 'feature_0', dep_path)
    print('SHAP plots saved:', shap_path, dep_path)
else:
    print('SHAP unavailable for selected model/runtime; permutation importance still generated.')
